# A mixed, incomplete table from inspection to reconstruction

A field team has measurements from 90 synthetic freshwater sites. Three variables are continuous and one records habitat type. Some cells were missed during sampling, but the team wants to retain the incomplete sites while exploring a low-dimensional summary.

This notebook follows one complete VBPCApy 0.5 workflow: inspect the table, encode mixed variables, fit the samples-by-features estimator with an observation mask, project the rows, and decode a reconstruction. The data are entirely synthetic and do not represent a real ecosystem.

## 1. Make the synthetic survey table

Rows are sites and columns are variables. Temperature is measured in degrees Celsius, nitrate in milligrams per litre, biomass in milligrams per cubic metre, and habitat is an unordered code: 0 = stream, 1 = pond, 2 = marsh. An unobserved synthetic gradient creates correlation among the variables; it is only a data-generation device, not a quantity fitted by VBPCApy. We then hide each cell independently with probability 0.12.

In [ ]:
import numpy as np

from vbpca_py import VBPCA, AutoEncoder, check_data

rng = np.random.default_rng(2026)
n_samples = 90
latent_gradient = rng.normal(size=n_samples)

temperature_c = 15.0 + 3.0 * latent_gradient + rng.normal(scale=0.8, size=n_samples)
nitrate_mg_l = 2.5 - 0.7 * latent_gradient + rng.normal(scale=0.25, size=n_samples)
biomass_mg_m3 = 8.0 + 1.8 * latent_gradient + rng.normal(scale=0.6, size=n_samples)
habitat_code = np.digitize(latent_gradient, [-0.6, 0.6]).astype(float)

complete_table = np.column_stack(
    [temperature_c, nitrate_mg_l, biomass_mg_m3, habitat_code]
)
observed = rng.random(complete_table.shape) >= 0.12
table = complete_table.copy()
table[~observed] = np.nan
column_names = [
    "temperature_c",
    "nitrate_mg_l",
    "biomass_mg_m3",
    "habitat_code",
]

print(f"Table shape: {table.shape[0]} sites x {table.shape[1]} variables")
print(f"Observed cells: {observed.sum()} of {observed.size}")
print(table[:5])

## 2. Inspect before fitting

`check_data` summarizes only cells marked observed. Its warning about the integer-coded habitat is useful: codes do not say whether a variable is ordered. We resolve that ambiguity explicitly in the encoder instead of treating 0, 1, and 2 as distances.

In [ ]:
report = check_data(table, mask=observed, column_names=column_names)
print(f"Preflight passed without warnings: {report.passed}")
for warning in report.warnings:
    print(f"- {warning}")

## 3. Encode the mixed variables

`AutoEncoder` consumes samples by variables, which is also the native VBPCA 0.5 orientation. We declare the first three variables continuous and habitat categorical. Continuous observations are standardized; habitat becomes a one-hot block. Missing input cells remain missing in the encoded array.

In [ ]:
encoder = AutoEncoder(
    column_types=[
        "continuous",
        "continuous",
        "continuous",
        "categorical",
    ],
    continuous_scaler="standard",
)
encoded = np.asarray(encoder.fit_transform(table, mask=observed), dtype=float)
encoded_observed = np.isfinite(encoded)

print(f"Encoded shape: {encoded.shape}")
print(encoder.get_feature_names_out())

## 4. Fit, project, and reconstruct

The native estimator keeps samples on rows; no transpose is needed. We use two components and eight fitting iterations because the notebook demonstrates the data path rather than capacity selection or convergence. The fitted `scores_` are joint variational means from training. `transform` instead computes fresh conditional scores for supplied rows under the fitted state; the two quantities need not agree. `inverse_transform` maps the conditional scores back to encoded variables.

In [ ]:
model = VBPCA(
    n_components=2,
    maxiters=8,
    niter_broadprior=0,
    verbose=0,
    random_state=2026,
    num_cpu=1,
    runtime_tuning="off",
)
model.fit(encoded, mask=encoded_observed)
joint_training_scores = model.scores_.copy()
conditional_scores = model.transform(encoded, mask=encoded_observed)
reconstructed_encoded = model.inverse_transform(conditional_scores)
reconstructed_table = encoder.inverse_transform(reconstructed_encoded)

print(f"Joint training score shape: {joint_training_scores.shape}")
print(f"Conditional score shape: {conditional_scores.shape}")
print(f"Fitted encoded-feature count: {model.n_features_in_}")
print(f"Fitting iterations: {model.n_iter_}")
print(f"Reconstructed shape: {reconstructed_table.shape}")
print(reconstructed_table[:5])

## 5. Check only the synthetic hidden cells

Because we retained the complete synthetic table, we can score the cells hidden above. These numbers describe this one simulation with independent random masking. They do not establish performance under nonrandom missingness, select a rank, demonstrate numerical convergence, or provide calibrated uncertainty.

In [ ]:
continuous_columns = np.array([0, 1, 2])
missing_continuous = ~observed[:, continuous_columns]
continuous_truth = complete_table[:, continuous_columns]
continuous_prediction = np.asarray(
    reconstructed_table[:, continuous_columns], dtype=float
)
continuous_units = {
    "temperature_c": "degrees Celsius",
    "nitrate_mg_l": "mg/L",
    "biomass_mg_m3": "mg/m^3",
}
continuous_mae = {
    column_names[column]: float(
        np.mean(
            np.abs(
                continuous_prediction[~observed[:, column], column]
                - continuous_truth[~observed[:, column], column]
            )
        )
    )
    for column in continuous_columns
}

missing_habitat = ~observed[:, 3]
habitat_prediction = np.asarray(reconstructed_table[:, 3], dtype=float)
habitat_accuracy = np.mean(
    habitat_prediction[missing_habitat] == complete_table[missing_habitat, 3]
)

for variable, value in continuous_mae.items():
    print(f"{variable} MAE: {value:.3f} {continuous_units[variable]}")
print(f"Habitat accuracy: {habitat_accuracy:.1%}")

## What to carry into your own analysis

Keep samples on rows for the native 0.5 estimator and its preprocessors. Declare categorical or ordinal meaning instead of asking numeric codes to imply it. Keep a Boolean observation mask alongside the table, and design validation to match the missingness or prediction question you actually care about. VBPCApy conditions on the supplied observation pattern; it does not model why values are missing.